# Baylo in-house models

Three small models trained on Baylo's own listings, to replace the paid AI calls:

| | Model | Replaces |
|---|---|---|
| **A** | Photo → category (MobileNetV3, transfer learning) | Claude item identify |
| **B** | Value estimate (Ridge regression on title, category, condition, photo) | Claude price check (as a fallback; it cannot search live prices) |
| **C** | Duplicate photo check (photo embeddings + cosine similarity) | Claude stage-2 double-check |

**How to run:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**. When asked, upload `items.csv` (from `scripts/export-training-data.ts`). At the end a zip with the models and charts downloads.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import os, io, json, math, random, time, zipfile, concurrent.futures as cf
import numpy as np, pandas as pd, requests
from PIL import Image
import torch, torch.nn as nn, torchvision
from torchvision import transforms as T
import matplotlib; matplotlib.use('Agg') if 'google.colab' not in str(get_ipython()) else None
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT = 'baylo-models'; IMG_DIR = 'img'
os.makedirs(OUT, exist_ok=True); os.makedirs(IMG_DIR, exist_ok=True)
IN_COLAB = 'google.colab' in str(get_ipython())
QUICK = os.environ.get('BAYLO_QUICK') == '1'   # tiny smoke-test mode (used to test this notebook offline)
print('device:', DEVICE, '| torch', torch.__version__, '| colab:', IN_COLAB, '| quick:', QUICK)

# Settings you may change
USE_DEMO_LISTINGS = True   # include demo/seed listings in training (they are real photos with real labels)
EPOCHS_HEAD, EPOCHS_FINE = (1, 1) if QUICK else (6, 14)
BATCH = 32

In [ ]:
# ── Load items.csv ────────────────────────────────────────────────────────────
CSV = 'items.csv'
if IN_COLAB and not os.path.exists(CSV):
    from google.colab import files
    up = files.upload()            # choose items.csv
    CSV = next(iter(up))
df = pd.read_csv(CSV)
if not USE_DEMO_LISTINGS:
    df = df[df.is_demo == 0]
df = df.dropna(subset=['image_url', 'category']).reset_index(drop=True)
print(len(df), 'photos from', df.item_id.nunique(), 'listings')
print(df.groupby('category').item_id.nunique().sort_values(ascending=False).to_string())

In [ ]:
# ── Download photos (public Cloudinary URLs), 256 px ─────────────────────────
def thumb(url):
    # ask Cloudinary for a small JPEG instead of the full photo
    return url.replace('/upload/', '/upload/w_320,h_320,c_limit,q_80,f_jpg/') if 'res.cloudinary.com' in url else url

def fetch(row):
    path = f"{IMG_DIR}/{row.item_id}_{row.photo_index}.jpg"
    if os.path.exists(path): return path
    try:
        r = requests.get(thumb(row.image_url), timeout=20); r.raise_for_status()
        im = Image.open(io.BytesIO(r.content)).convert('RGB'); im.thumbnail((256, 256)); im.save(path, quality=90)
        return path
    except Exception as e:
        return None

with cf.ThreadPoolExecutor(16) as ex:
    df['path'] = list(ex.map(fetch, df.itertuples()))
print('failed downloads:', df.path.isna().sum())
df = df.dropna(subset=['path']).reset_index(drop=True)

In [ ]:
# ── Train/validation split BY LISTING (photos of one listing never sit on both sides) ──
from sklearn.model_selection import train_test_split
items = df.groupby('item_id').agg(category=('category', 'first')).reset_index()
counts = items.category.value_counts()
strat = items.category.where(items.category.map(counts) >= 2, 'RARE')   # stratify where possible
tr_ids, va_ids = train_test_split(items.item_id, test_size=0.2, random_state=SEED, stratify=strat if strat.value_counts().min() >= 2 else None)
df['split'] = np.where(df.item_id.isin(set(va_ids)), 'val', 'train')
CATS = sorted(df.category.unique()); C2I = {c: i for i, c in enumerate(CATS)}
df['y'] = df.category.map(C2I)
print(df.split.value_counts().to_string(), '\n', len(CATS), 'categories:', CATS)

BRACKET_CEILINGS = [100, 250, 500, 900, 1500, 2500, 4000, 9000, 12000]   # src/lib/brackets.ts
def bracket_of(v): return 1 + sum(v > c for c in BRACKET_CEILINGS)

## A. Photo → category

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
tf_train = T.Compose([T.RandomResizedCrop(224, scale=(0.6, 1)), T.RandomHorizontalFlip(), T.ColorJitter(0.25, 0.25, 0.2), T.ToTensor(), T.Normalize(MEAN, STD)])
tf_eval = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)])

class Photos(torch.utils.data.Dataset):
    def __init__(self, frame, tf): self.f, self.tf = frame.reset_index(drop=True), tf
    def __len__(self): return len(self.f)
    def __getitem__(self, i):
        r = self.f.iloc[i]; return self.tf(Image.open(r.path).convert('RGB')), int(r.y)

train_df, val_df = df[df.split == 'train'], df[df.split == 'val']
dl_tr = torch.utils.data.DataLoader(Photos(train_df, tf_train), batch_size=BATCH, shuffle=True, num_workers=2)
dl_va = torch.utils.data.DataLoader(Photos(val_df, tf_eval), batch_size=BATCH, num_workers=2)

class BayloNet(nn.Module):
    # MobileNetV3-Large backbone; returns (category logits, 960-d L2-normalised photo embedding)
    def __init__(self, n):
        super().__init__()
        m = torchvision.models.mobilenet_v3_large(weights=torchvision.models.MobileNet_V3_Large_Weights.DEFAULT)
        self.features, self.pool = m.features, m.avgpool
        self.head = nn.Sequential(nn.Linear(960, 512), nn.Hardswish(), nn.Dropout(0.3), nn.Linear(512, n))
    def forward(self, x):
        e = torch.flatten(self.pool(self.features(x)), 1)
        return self.head(e), nn.functional.normalize(e, dim=1)

model = BayloNet(len(CATS)).to(DEVICE)
freq = train_df.y.value_counts().reindex(range(len(CATS)), fill_value=0).values
w = torch.tensor(1.0 / np.maximum(freq, 1), dtype=torch.float32); w = w / w.mean()
crit = nn.CrossEntropyLoss(weight=w.to(DEVICE), label_smoothing=0.1)

def run_epoch(train, opt=None):
    model.train(train); tot = correct = n = 0
    for x, y in (dl_tr if train else dl_va):
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.set_grad_enabled(train):
            logits, _ = model(x); loss = crit(logits, y)
            if train: opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item() * len(y); correct += (logits.argmax(1) == y).sum().item(); n += len(y)
    return tot / max(n, 1), correct / max(n, 1)

best, best_state, hist = -1, None, []
for phase, epochs, lr, freeze in (('head', EPOCHS_HEAD, 1e-3, True), ('fine-tune', EPOCHS_FINE, 2e-4, False)):
    for p in model.features.parameters(): p.requires_grad = not freeze
    if not freeze:   # fine-tune only the last blocks; the early ones keep ImageNet features
        for p in model.features[:10].parameters(): p.requires_grad = False
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, max(epochs, 1))
    for ep in range(epochs):
        trl, tra = run_epoch(True, opt); val, vaa = run_epoch(False); sched.step()
        hist.append((phase, ep + 1, trl, tra, val, vaa))
        print(f'{phase:9s} {ep+1:2d}  train loss {trl:.3f} acc {tra:.2%}   val loss {val:.3f} acc {vaa:.2%}')
        if vaa > best: best, best_state = vaa, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state); print(f'best validation accuracy: {best:.2%}')

In [ ]:
# ── Evaluate A ────────────────────────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix
model.eval(); P, Y, EMB = [], [], []
with torch.no_grad():
    for x, y in dl_va:
        logits, e = model(x.to(DEVICE)); P.append(torch.softmax(logits, 1).cpu()); Y.append(y); EMB.append(e.cpu())
P, Y = torch.cat(P).numpy(), torch.cat(Y).numpy()
top1 = (P.argmax(1) == Y).mean(); top3 = np.mean([y in np.argsort(-p)[:3] for p, y in zip(P, Y)])
print(f'Validation photos: {len(Y)}   top-1 accuracy: {top1:.2%}   top-3 accuracy: {top3:.2%}\n')
labels_present = sorted(set(Y) | set(P.argmax(1)))
print(classification_report(Y, P.argmax(1), labels=labels_present, target_names=[CATS[i] for i in labels_present], zero_division=0))
cm = confusion_matrix(Y, P.argmax(1), labels=range(len(CATS)))
fig, ax = plt.subplots(figsize=(9, 8)); ax.imshow(cm, cmap='Greens')
ax.set_xticks(range(len(CATS))); ax.set_xticklabels(CATS, rotation=90); ax.set_yticks(range(len(CATS))); ax.set_yticklabels(CATS)
for i in range(len(CATS)):
    for j in range(len(CATS)):
        if cm[i, j]: ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=8)
ax.set_xlabel('predicted'); ax.set_ylabel('actual'); ax.set_title(f'Photo → category (top-1 {top1:.0%}, top-3 {top3:.0%})')
plt.tight_layout(); plt.savefig(f'{OUT}/A_confusion_matrix.png', dpi=160); plt.show()

In [ ]:
# ── Export A to ONNX (logits + embedding), and check it matches PyTorch ───────
import onnx, onnxruntime as ort
model.cpu().eval()
dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(model, dummy, f'{OUT}/baylo-classifier.onnx', input_names=['image'], output_names=['logits', 'embedding'],
                  dynamic_axes={'image': {0: 'n'}, 'logits': {0: 'n'}, 'embedding': {0: 'n'}}, opset_version=18)
# newer PyTorch writes the weights to a side file; fold them into ONE .onnx file so it deploys as a single asset
m_onnx = onnx.load(f'{OUT}/baylo-classifier.onnx')
onnx.save_model(m_onnx, f'{OUT}/baylo-classifier.onnx', save_as_external_data=False)
if os.path.exists(f'{OUT}/baylo-classifier.onnx.data'): os.remove(f'{OUT}/baylo-classifier.onnx.data')
sess = ort.InferenceSession(f'{OUT}/baylo-classifier.onnx')
o_logits, o_emb = sess.run(None, {'image': dummy.numpy()})
t_logits, t_emb = model(dummy)
print('max |ONNX - PyTorch|:', float(np.abs(o_logits - t_logits.detach().numpy()).max()), '| size MB:', round(os.path.getsize(f'{OUT}/baylo-classifier.onnx') / 1e6, 1))
json.dump({'categories': CATS, 'input': 224, 'resize': 256, 'mean': MEAN, 'std': STD, 'val_top1': float(top1), 'val_top3': float(top3)},
          open(f'{OUT}/labels.json', 'w'), indent=2)
model.to(DEVICE)

## C. Duplicate photo check (photo embeddings)

In [ ]:
# Same photo, edited (crop / resize / recompress / brighten) should MATCH; different listings should NOT.
import random
from sklearn.metrics import roc_curve, roc_auc_score
def edit(im):
    w, h = im.size; s = random.uniform(0.75, 0.95)
    x0, y0 = random.randint(0, int(w * (1 - s))), random.randint(0, int(h * (1 - s)))
    im = im.crop((x0, y0, x0 + int(w * s), y0 + int(h * s))).resize((random.randint(180, 256),) * 2)
    im = T.ColorJitter(0.3, 0.3)(im)
    buf = io.BytesIO(); im.save(buf, 'JPEG', quality=random.randint(35, 80)); return Image.open(buf).convert('RGB')

@torch.no_grad()
def embed(ims):
    model.eval(); return model(torch.stack([tf_eval(i) for i in ims]).to(DEVICE))[1].cpu().numpy()

pool = val_df.drop_duplicates('item_id').sample(min(len(val_df.drop_duplicates('item_id')), 300), random_state=SEED)
orig = [Image.open(p).convert('RGB') for p in pool.path]
E0, E1 = embed(orig), embed([edit(i) for i in orig])
pos = (E0 * E1).sum(1)
ii = np.random.randint(0, len(E0), 4 * len(E0)); jj = np.random.randint(0, len(E0), 4 * len(E0)); keep = pool.item_id.values[ii] != pool.item_id.values[jj]
neg = (E0[ii[keep]] * E0[jj[keep]]).sum(1)
yv = np.r_[np.ones_like(pos), np.zeros_like(neg)]; sv = np.r_[pos, neg]
auc = roc_auc_score(yv, sv); fpr, tpr, thr = roc_curve(yv, sv)
k = np.where(fpr <= 0.01)[0][-1]; THRESH = float(thr[k])
print(f'pairs: {len(pos)} same-photo, {len(neg)} different  |  ROC AUC {auc:.3f}')
print(f'threshold {THRESH:.3f}: catches {tpr[k]:.1%} of edited copies at {fpr[k]:.1%} false matches')
plt.figure(figsize=(5, 5)); plt.plot(fpr, tpr, color='#1F4D3A'); plt.plot([0, 1], [0, 1], ':', color='grey')
plt.scatter([fpr[k]], [tpr[k]], color='#C9A54A', zorder=3); plt.xlabel('false match rate'); plt.ylabel('edited copies caught')
plt.title(f'Duplicate check ROC (AUC {auc:.3f})'); plt.tight_layout(); plt.savefig(f'{OUT}/C_duplicate_roc.png', dpi=160); plt.show()
json.dump({'cosine_threshold': THRESH, 'auc': float(auc), 'tpr': float(tpr[k]), 'fpr': float(fpr[k])}, open(f'{OUT}/dup-threshold.json', 'w'), indent=2)

## B. Value estimate

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.decomposition import TruncatedSVD
from scipy import sparse

CONDS = ['NEW', 'LIKE_NEW', 'GOOD', 'FAIR', 'POOR']
COND_MULT = {'NEW': 1.40, 'LIKE_NEW': 1.20, 'GOOD': 1.00, 'FAIR': 0.70, 'POOR': 0.45}   # src/lib/valuation.ts
BANDS = {'ELECTRONICS': (500, 10000), 'CLOTHING': (100, 1000), 'BAGS': (200, 2000), 'BEAUTY': (100, 1000), 'ACCESSORIES': (100, 1500),
         'FURNITURE': (500, 6000), 'BOOKS': (50, 500), 'GAMING': (500, 8000), 'SPORTS': (200, 3000), 'BIKES': (1500, 12000), 'TOYS': (100, 1000),
         'TOOLS': (200, 3000), 'MUSIC': (500, 8000), 'ART': (100, 2000), 'COLLECTIBLES': (100, 3000), 'PETS': (100, 1000), 'PLANTS': (50, 600),
         'FOOD': (50, 300), 'OTHER': (100, 2000)}

it = df[(df.is_perishable == 0) & df.value.notna() & (df.value > 0)].copy()
emb_by_item = {}
with torch.no_grad():
    for item_id, g in it.groupby('item_id'):
        emb_by_item[item_id] = embed([Image.open(p).convert('RGB') for p in g.path]).mean(0)
it = it.drop_duplicates('item_id').reset_index(drop=True)
it['logv'] = np.log(it.value); tr, va = it[it.split == 'train'], it[it.split == 'val']
print(len(tr), 'train listings,', len(va), 'validation listings')

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_features=3000, sublinear_tf=True).fit(tr.title)
def X_of(f, with_img):
    cat = sparse.csr_matrix(pd.get_dummies(f.category).reindex(columns=CATS, fill_value=0).values.astype(float))
    cond = sparse.csr_matrix(np.array([[CONDS.index(c) == k for k in range(5)] for c in f.condition], dtype=float))
    parts = [tfidf.transform(f.title), cat, cond]
    if with_img: parts.append(sparse.csr_matrix(np.stack([emb_by_item[i] for i in f.item_id])))
    return sparse.hstack(parts).tocsr()

def rule_baseline(f):   # Baylo today: average condition-normalised value in the category (>=3 comparables), else band midpoint, x condition
    out = []
    for c, cond in zip(f.category, f.condition):
        comp = tr[tr.category == c]
        base = (comp.value / comp.condition.map(COND_MULT)).mean() if len(comp) >= 3 else sum(BANDS.get(c, (50, 1000))) / 2
        out.append(max(1, base * COND_MULT.get(cond, 1)))
    return np.array(out)

def scores(name, pred):
    true = va.value.values; pred = np.maximum(1, pred)
    tb, pb = np.array([bracket_of(v) for v in true]), np.array([bracket_of(v) for v in pred])
    return {'model': name, 'median error (points)': float(np.median(np.abs(pred - true))), 'median % error': float(np.median(np.abs(pred - true) / true)),
            'same bracket': float((tb == pb).mean()), 'within one bracket': float((np.abs(tb - pb) <= 1).mean())}

alphas = np.logspace(-2, 3, 20)
ridge_txt = RidgeCV(alphas=alphas).fit(X_of(tr, False), tr.logv)
ridge_img = RidgeCV(alphas=alphas).fit(X_of(tr, True), tr.logv)
svd = TruncatedSVD(min(64, X_of(tr, True).shape[1] - 1), random_state=SEED).fit(X_of(tr, True))
gbm = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=SEED).fit(svd.transform(X_of(tr, True)), tr.logv)
res = pd.DataFrame([scores("Baylo's current rule", rule_baseline(va)),
                    scores('Ridge (title+category+condition)', np.exp(ridge_txt.predict(X_of(va, False)))),
                    scores('Ridge (+ photo)', np.exp(ridge_img.predict(X_of(va, True)))),
                    scores('Gradient boosting (+ photo)', np.exp(gbm.predict(svd.transform(X_of(va, True)))))])
print(res.to_string(index=False, float_format=lambda x: f'{x:.2f}'))
ax = res.set_index('model')[['same bracket', 'within one bracket']].plot.barh(figsize=(8, 3.5), color=['#1F4D3A', '#C9A54A'])
ax.set_xlim(0, 1); ax.set_title('Value estimate: bracket accuracy on validation listings'); plt.tight_layout(); plt.savefig(f'{OUT}/B_value_brackets.png', dpi=160); plt.show()
res.to_csv(f'{OUT}/B_value_results.csv', index=False)

In [ ]:
# ── Export the better Ridge model as plain JSON (the server runs it in TypeScript, no Python needed) ──
score = lambda row: row['within one bracket'] + row['same bracket']
use_img = score(res.iloc[2]) > score(res.iloc[1])
if max(score(res.iloc[1]), score(res.iloc[2])) <= score(res.iloc[0]):
    print('NOTE: neither Ridge model beats the current rule on this data yet; keep the rule live and collect more listings.')
r = ridge_img if use_img else ridge_txt
coef = r.coef_; nt = len(tfidf.vocabulary_)
model_json = {
    'kind': 'ridge-log-value', 'uses_photo_embedding': bool(use_img), 'intercept': float(r.intercept_), 'alpha': float(r.alpha_),
    'tfidf': {'vocabulary': {k: int(v) for k, v in tfidf.vocabulary_.items()}, 'idf': tfidf.idf_.round(6).tolist(), 'ngram_range': [1, 2], 'sublinear_tf': True, 'lowercase': True,
              'token_pattern': r'(?u)\b\w\w+\b'},
    'coef_text': coef[:nt].round(6).tolist(), 'coef_category': dict(zip(CATS, coef[nt:nt + len(CATS)].round(6).tolist())),
    'coef_condition': dict(zip(CONDS, coef[nt + len(CATS):nt + len(CATS) + 5].round(6).tolist())),
    'coef_embedding': coef[nt + len(CATS) + 5:].round(6).tolist() if use_img else None,
    'validation': res.iloc[2 if use_img else 1].to_dict(),
}
json.dump(model_json, open(f'{OUT}/value-model.json', 'w'))
print('exported', 'Ridge (+ photo)' if use_img else 'Ridge (text)', '| size KB:', round(os.path.getsize(f'{OUT}/value-model.json') / 1e3))

In [ ]:
# ── Summary + download everything ─────────────────────────────────────────────
pd.DataFrame(hist, columns=['phase', 'epoch', 'train_loss', 'train_acc', 'val_loss', 'val_acc']).to_csv(f'{OUT}/A_training_history.csv', index=False)
summary = {'listings': int(df.item_id.nunique()), 'photos': int(len(df)), 'categories': len(CATS),
           'A_top1': float(top1), 'A_top3': float(top3), 'C_auc': float(auc), 'C_threshold': THRESH,
           'B': res.to_dict(orient='records')}
json.dump(summary, open(f'{OUT}/summary.json', 'w'), indent=2); print(json.dumps(summary, indent=2))
with zipfile.ZipFile('baylo-models.zip', 'w') as z:
    for f in os.listdir(OUT): z.write(f'{OUT}/{f}', f)
if IN_COLAB:
    from google.colab import files; files.download('baylo-models.zip')